# Qwen2-VL QLoRA fine-tune on Kaggle GPU

**Before running:** Notebook menu → Settings → **Accelerator = GPU T4 x2** (or P100), and **Internet = On**.

This notebook clones your code from GitHub and runs `train.py` on the GPU. Checkpoints land in `/kaggle/working` and survive to the output tab.

In [ ]:
# 1) Sanity-check the GPU
!nvidia-smi

In [ ]:
# 2) Clone your repo (public).
REPO = 'https://github.com/smshaqib/qwen2vl-finetune.git'

import os
if os.path.exists('/kaggle/working/repo'):
    !cd /kaggle/working/repo && git pull
else:
    !git clone $REPO /kaggle/working/repo

%cd /kaggle/working/repo

In [ ]:
# 3) Install deps not already on the Kaggle image (quiet).
!pip install -q -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils

In [ ]:
# 4) (Optional) HF login if you later use a gated model/dataset.
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# login(UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
# 5) Smoke test: train on 8 samples (~1-2 min) to shake out errors cheaply.
#    Once this passes, switch to: --config config.yaml  for the real run.
!python train.py --config config.debug.yaml

In [ ]:
# 6) Quick inference sanity-check with the trained (debug) adapter.
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel
from qwen_vl_utils import process_vision_info

BASE = 'Qwen/Qwen2-VL-2B-Instruct'
ADAPTER = '/kaggle/working/qwen2vl-lora-debug'   # debug run output; use qwen2vl-lora for the real run

proc = AutoProcessor.from_pretrained(ADAPTER)
model = Qwen2VLForConditionalGeneration.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(model, ADAPTER)

msgs = [{'role':'user','content':[
    {'type':'image','image':'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg'},
    {'type':'text','text':'What is in this image?'}]}]
text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
imgs, vids = process_vision_info(msgs)
inp = proc(text=[text], images=imgs, videos=vids, return_tensors='pt').to(model.device)
out = model.generate(**inp, max_new_tokens=128)
print(proc.batch_decode(out[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0])